# Validation Stress Test — AI4I 2020 Predictive Maintenance

## Objective

Test how model performance changes under different validation assumptions.

We compare:

1. Stratified shuffled 5-fold CV
2. Genuine ordered/block 5-fold robustness testing

The dataset does not contain a real timestamp. Therefore the ordered evaluation is
a **robustness stress test**, not true temporal forecasting.

In [9]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("../data/raw/ai4i2020.csv")
df = pd.read_csv(DATA_PATH)

TARGET = "Machine failure"
RANDOM_STATE = 42

raw_features = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

engineered_features = raw_features + [
    "temp_diff",
    "power_w",
    "wear_torque",
    "strain_ratio"
]

# Feature engineering
df["temp_diff"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power_w"] = (
    df["Torque [Nm]"] * df["Rotational speed [rpm]"] * 2 * np.pi / 60
)
strain_limits = {"L": 11000, "M": 12000, "H": 13000}
df["wear_torque"] = df["Tool wear [min]"] * df["Torque [Nm]"]
df["strain_ratio"] = df["wear_torque"] / df["Type"].map(strain_limits)

# Final holdout is the last 20% by original row order.
split_point = int(len(df) * 0.8)
dev = df.iloc[:split_point].copy()
final_holdout = df.iloc[split_point:].copy()

print("Development:", dev.shape)
print("Final holdout:", final_holdout.shape)

Development: (8000, 18)
Final holdout: (2000, 18)


## 1. Model factory

In [10]:
def make_preprocessor(feature_columns):
    numeric = [c for c in feature_columns if c != "Type"]
    return ColumnTransformer(
        [
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Type"]),
            ("num", "passthrough", numeric)
        ]
    )

def make_models(feature_columns):
    models = {
        "LogReg": Pipeline([
            ("prep", make_preprocessor(feature_columns)),
            ("model", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),
        "RandomForest": Pipeline([
            ("prep", make_preprocessor(feature_columns)),
            ("model", RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced_subsample",
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ]),
        "HistGB": Pipeline([
            ("prep", make_preprocessor(feature_columns)),
            ("model", HistGradientBoostingClassifier(
                max_iter=300,
                learning_rate=0.05,
                random_state=RANDOM_STATE
            ))
        ])
    }

    try:
        from xgboost import XGBClassifier
        models["XGBoost"] = Pipeline([
            ("prep", make_preprocessor(feature_columns)),
            ("model", XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.9,
                colsample_bytree=0.9,
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                scale_pos_weight=1.0
            ))
        ])
    except ImportError:
        pass

    return models

## 2. Evaluation helper

Each fold is fitted independently. Metrics are calculated from the held-out fold.

In [11]:
def fold_metrics(y_true, score, threshold=0.5):
    pred = (score >= threshold).astype(int)
    return {
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, score),
        "average_precision": average_precision_score(y_true, score),
    }

def run_cv(model, X, y, splitter):
    records = []

    for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
        m = clone(model)
        m.fit(X.iloc[train_idx], y.iloc[train_idx])

        score = m.predict_proba(X.iloc[test_idx])[:, 1]

        metrics = fold_metrics(
            y.iloc[test_idx].to_numpy(),
            score
        )

        metrics["fold"] = fold
        records.append(metrics)

    return pd.DataFrame(records)

## 3. Stratified shuffled CV

This is the standard IID-like validation assumption.

In [12]:
stratified_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

stratified_records = []

for feature_name, feature_columns in [
    ("raw", raw_features),
    ("engineered", engineered_features)
]:
    y = dev[TARGET].astype(int).reset_index(drop=True)
    X = dev[feature_columns].reset_index(drop=True)

    for model_name, model in make_models(feature_columns).items():
        fold_df = run_cv(model, X, y, stratified_cv)
        fold_df["validation"] = "stratified_shuffled"
        fold_df["features"] = feature_name
        fold_df["model"] = model_name
        stratified_records.append(fold_df)

stratified_results = pd.concat(stratified_records, ignore_index=True)

stratified_summary = (
    stratified_results
    .groupby(["model", "features"])
    [["precision", "recall", "f1", "roc_auc", "average_precision"]]
    .agg(["mean", "std"])
)

stratified_summary

precision              recall                  f1  \
                             mean       std      mean       std      mean   
model        features                                                       
HistGB       engineered  0.949731  0.041002  0.823333  0.048016  0.881867   
             raw         0.873038  0.098021  0.656667  0.072265  0.749095   
LogReg       engineered  0.184791  0.004907  0.846667  0.021731  0.303339   
             raw         0.149100  0.005106  0.796667  0.038006  0.251157   
RandomForest engineered  0.944172  0.023037  0.843333  0.060782  0.890009   
             raw         0.815479  0.067421  0.590000  0.068313  0.683863   
XGBoost      engineered  0.932015  0.077644  0.810000  0.065192  0.866435   
             raw         0.874198  0.042199  0.646667  0.038006  0.743126   

                                    roc_auc           average_precision  \
                              std      mean       std              mean   
model        features                                                     
HistGB       engineered  0.043837  0.976846  0.014374          0.902181   
             raw         0.080268  0.973807  0.014520          0.830168   
LogReg       engineered  0.007218  0.926039  0.008595          0.451438   
             raw         0.008697  0.894812  0.010461          0.438124   
RandomForest engineered  0.037631  0.978571  0.012160          0.906620   
             raw         0.066125  0.970733  0.013609          0.767257   
XGBoost      engineered  0.068395  0.980338  0.012342          0.903334   
             raw         0.036870  0.977636  0.012183          0.825805   

                                   
                              std  
model        features              
HistGB       engineered  0.044783  
             raw         0.076832  
LogReg       engineered  0.045212  
             raw         0.056992  
RandomForest engineered  0.034961  
             raw         0.063084  
XGBoost      engineered  0.050266  
             raw         0.065111

## 4. Ordered/block robustness test

The development set is still in original UDI order.

KFold without shuffling therefore creates five contiguous validation blocks.

This is intentionally described as a block robustness test, not forecasting.

In [13]:
block_cv = KFold(
    n_splits=5,
    shuffle=False
)

block_records = []

for feature_name, feature_columns in [
    ("raw", raw_features),
    ("engineered", engineered_features)
]:
    y = dev[TARGET].astype(int).reset_index(drop=True)
    X = dev[feature_columns].reset_index(drop=True)

    for model_name, model in make_models(feature_columns).items():
        fold_df = run_cv(model, X, y, block_cv)
        fold_df["validation"] = "ordered_block"
        fold_df["features"] = feature_name
        fold_df["model"] = model_name
        block_records.append(fold_df)

block_results = pd.concat(block_records, ignore_index=True)

block_summary = (
    block_results
    .groupby(["model", "features"])
    [["precision", "recall", "f1", "roc_auc", "average_precision"]]
    .agg(["mean", "std"])
)

block_summary

precision              recall                  f1  \
                             mean       std      mean       std      mean   
model        features                                                       
HistGB       engineered  0.945391  0.037961  0.638355  0.219433  0.739154   
             raw         0.754778  0.213405  0.466371  0.153183  0.547748   
LogReg       engineered  0.200162  0.110318  0.758856  0.072102  0.300023   
             raw         0.250679  0.278845  0.715408  0.113448  0.302626   
RandomForest engineered  0.975877  0.035564  0.648357  0.235974  0.752237   
             raw         0.759204  0.303456  0.430799  0.149983  0.491536   
XGBoost      engineered  0.898782  0.044979  0.690996  0.170208  0.767743   
             raw         0.703991  0.253849  0.510316  0.206105  0.528403   

                                    roc_auc           average_precision  \
                              std      mean       std              mean   
model        features                                                     
HistGB       engineered  0.193415  0.964165  0.019638          0.854893   
             raw         0.147932  0.929007  0.051905          0.620078   
LogReg       engineered  0.127482  0.901191  0.039342          0.404259   
             raw         0.223563  0.871061  0.067305          0.468323   
RandomForest engineered  0.212606  0.967216  0.013722          0.861665   
             raw         0.163371  0.946388  0.024052          0.634501   
XGBoost      engineered  0.122911  0.969564  0.023903          0.864761   
             raw         0.166888  0.939647  0.043341          0.644734   

                                   
                              std  
model        features              
HistGB       engineered  0.048267  
             raw         0.155210  
LogReg       engineered  0.100706  
             raw         0.141706  
RandomForest engineered  0.048150  
             raw         0.112500  
XGBoost      engineered  0.053916  
             raw         0.110846

## 5. Compare validation strategies

In [14]:
combined = pd.concat(
    [stratified_results, block_results],
    ignore_index=True
)

comparison = (
    combined
    .groupby(["validation", "model", "features"])
    [["precision", "recall", "f1", "roc_auc", "average_precision"]]
    .mean()
    .reset_index()
)

comparison.sort_values(
    ["features", "average_precision"],
    ascending=[True, False]
)

,validation,model,features,precision,recall,f1,roc_auc,average_precision
12,stratified_shuffled,RandomForest,engineered,0.944172,0.843333,0.890009,0.978571,0.906620
14,stratified_shuffled,XGBoost,engineered,0.932015,0.810000,0.866435,0.980338,0.903334
8,stratified_shuffled,HistGB,engineered,0.949731,0.823333,0.881867,0.976846,0.902181
6,ordered_block,XGBoost,engineered,0.898782,0.690996,0.767743,0.969564,0.864761
4,ordered_block,RandomForest,engineered,0.975877,0.648357,0.752237,0.967216,0.861665
0,ordered_block,HistGB,engineered,0.945391,0.638355,0.739154,0.964165,0.854893
10,stratified_shuffled,LogReg,engineered,0.184791,0.846667,0.303339,0.926039,0.451438
2,ordered_block,LogReg,engineered,0.200162,0.758856,0.300023,0.901191,0.404259
9,stratified_shuffled,HistGB,raw,0.873038,0.656667,0.749095,0.973807,0.830168
15,stratified_shuffled,XGBoost,raw,0.874198,0.646667,0.743126,0.977636,0.825805


## 6. Robustness gap

A positive gap means shuffled validation produced a higher score than ordered/block
validation for the same model and feature set.

In [15]:
wide = (
    comparison
    .pivot_table(
        index=["model", "features"],
        columns="validation",
        values="average_precision"
    )
    .reset_index()
)

if {"stratified_shuffled", "ordered_block"}.issubset(wide.columns):
    wide["AP_gap_shuffle_minus_block"] = (
        wide["stratified_shuffled"]
        - wide["ordered_block"]
    )

wide

validation,model,features,ordered_block,stratified_shuffled,AP_gap_shuffle_minus_block
0,HistGB,engineered,0.854893,0.902181,0.047287
1,HistGB,raw,0.620078,0.830168,0.210089
2,LogReg,engineered,0.404259,0.451438,0.047179
3,LogReg,raw,0.468323,0.438124,-0.030199
4,RandomForest,engineered,0.861665,0.906620,0.044955
5,RandomForest,raw,0.634501,0.767257,0.132756
6,XGBoost,engineered,0.864761,0.903334,0.038573
7,XGBoost,raw,0.644734,0.825805,0.181072


## 7. Final holdout definition

The final holdout has never been used for model selection, feature selection or
threshold tuning in this notebook.

It is retained for Notebook 08.

In [16]:
print("Final holdout rows:", len(final_holdout))
print("Final holdout failures:", int(final_holdout[TARGET].sum()))
print("Final holdout failure rate:", round(final_holdout[TARGET].mean() * 100, 3), "%")

Final holdout rows: 2000
Final holdout failures: 39
Final holdout failure rate: 1.95 %


## Conclusion

This notebook answers whether the reported ML performance is stable across two
different validation assumptions.

Any difference is interpreted as a robustness result. It is not described as proof
of temporal forecasting because the dataset does not provide a genuine timestamp.